# 07 - Full multimodal features for run_02 (port of lab notebook 3, M3)

Builds the lab's 524-number encounter vector for ALL run_02 inpatient stays:
profile (age, gender, race, ethnicity, has_vitals, 5 vitals) + condition, procedure, note and medication
Word2Vec embeddings (128 each) + lab / medication activity flags.

Changes vs the lab notebook (on purpose):
- Cohort = all inpatient stays (same as our notebook 01), not the pure-PLOS cohort.
- Uses our patient-level train/val/test split from notebook 01. Word2Vec models are fit on TRAIN patients only.
- Conditions = codes recorded at the stay PLUS conditions already active at admission.
- LLM steps (Groq: procedure to ICD-10-PCS, note to ICD-10-CM) are optional switches, OFF by default.
  When off, procedures use their SNOMED codes. Turn on only with GROQ_API_KEY in Colab Secrets.
- scispaCy abbreviation expansion is optional (slow on 20k notes), OFF by default.

Resumable: each stage saves to Drive and is skipped if its file exists.
**Admission window (`WINDOW_HOURS`, Cell 1).** `None` = whole stay (writes `features_full/`). `24` = admission-time
version (writes `features_full_24h/`): vitals = latest reading up to 24 h after admission (including earlier visits);
labs, procedures, medication starts = first 24 h only; medications also include those already active at admission;
conditions recorded at the stay = first 24 h only; note = the one dated on the admission day.

Run: Runtime > Run all. CPU is fine.

In [ ]:
# Cell 1 - settings
from google.colab import drive
drive.mount('/content/drive')
!pip -q install gensim

BASE      = "/content/drive/MyDrive/genai_synthea/run_02"
RAW       = f"{BASE}/raw/csv"
NOTES_TAR = f"{BASE}/raw/notes.tar.gz"
NOTES_DIR = "/content/notes_local/notes"
GENAI     = f"{BASE}/genai"                 # notebook 01 outputs (split + threshold)
WINDOW_HOURS = 24            # None = whole stay (original run); 24 = only what is known in the first 24 h
OUT       = f"{BASE}/features_full" + (f"_{WINDOW_HOURS}h" if WINDOW_HOURS else "")   # separate folder per setting

USE_LLM_PROCEDURES = False   # Groq: procedure description -> ICD-10-PCS
USE_LLM_NOTES      = False   # Groq: note -> ICD-10-CM codes (slow: ~1 call per stay)
USE_SCISPACY       = False   # abbreviation expansion (slow)
CHUNK = 200_000
W2V_DIM, SEED = 128, 105

import os, re, json, pickle, time, numpy as np, pandas as pd
from datetime import timedelta
os.makedirs(OUT, exist_ok=True)
def done(name): return os.path.exists(f"{OUT}/{name}")
def save(obj, name): pickle.dump(obj, open(f"{OUT}/{name}", "wb"))
def load(name): return pickle.load(open(f"{OUT}/{name}", "rb"))

In [ ]:
# Cell 2 - cohort: all inpatient stays, our threshold and our split
ENC_COLS = ['Id', 'PATIENT', 'ENCOUNTERCLASS', 'START', 'STOP', 'DESCRIPTION', 'REASONDESCRIPTION']
enc = pd.concat([c[c.ENCOUNTERCLASS == 'inpatient'] for c in
                 pd.read_csv(f"{RAW}/encounters.csv", usecols=ENC_COLS, chunksize=CHUNK)], ignore_index=True)
enc = enc.rename(columns={'Id': 'hadm_id', 'PATIENT': 'subject_id', 'START': 'admittime', 'STOP': 'dischtime',
                          'DESCRIPTION': 'encounter_type', 'REASONDESCRIPTION': 'encounter_reason'})
pat = pd.read_csv(f"{RAW}/patients.csv", usecols=['Id', 'GENDER', 'RACE', 'ETHNICITY', 'BIRTHDATE'])
pat = pat.rename(columns={'Id': 'subject_id', 'GENDER': 'gender', 'RACE': 'race', 'ETHNICITY': 'ethnicity', 'BIRTHDATE': 'birthdate'})
df = enc.merge(pat, on='subject_id', how='left')
for c in ('admittime', 'dischtime', 'birthdate'):
    df[c] = pd.to_datetime(df[c], errors='coerce', utc=True).dt.tz_localize(None)

# keep exactly the stays notebook 01 kept, with its split and threshold
meta = json.load(open(f"{GENAI}/meta.json"))
THR = meta['plos_threshold']
split = pd.concat([pd.read_parquet(f"{GENAI}/{s}.parquet", columns=['hadm_id', 'subject_id', 'target']).assign(split=s)
                   for s in ['train', 'val', 'test']])
df = df.merge(split[['hadm_id', 'split', 'target']], on='hadm_id', how='inner')
df['anchor_age'] = (df['admittime'] - df['birthdate']).dt.days // 365
df['is_plos'] = (df['target'] > THR).astype(int)
cohort = set(df['hadm_id']); train_subj = set(df.loc[df.split == 'train', 'subject_id'])
print(f"Stays {len(df):,} | patients {df.subject_id.nunique():,} | threshold {THR:.2f} d | PLOS {df.is_plos.mean():.1%}")
print(df['split'].value_counts().to_string())

In [ ]:
# Cell 3 - vitals snapshot + lab activity flag (observations, chunked)
VITALS = {'Heart rate': 'hr', 'Systolic Blood Pressure': 'sbp', 'Diastolic Blood Pressure': 'dbp',
          'Respiratory rate': 'rr', 'Body Weight': 'weight'}
CHANNELS = ['hr', 'sbp', 'dbp', 'rr', 'weight']
LAB_SKIP = set(VITALS) | {'Body Height', 'Body mass index (BMI) [Ratio]',
            'Pain severity - 0-10 verbal numeric rating [Score] - Reported', 'DALY', 'QOLS', 'QALY'}
df['wend'] = df['admittime'] + pd.Timedelta(hours=WINDOW_HOURS) if WINDOW_HOURS else df['dischtime']
df['wend'] = df[['wend', 'dischtime']].min(axis=1)            # never past discharge
subjects = set(df.subject_id)
if not done("vitals_labs.pkl"):
    t0 = time.time(); parts = []
    for c in pd.read_csv(f"{RAW}/observations.csv", usecols=['PATIENT', 'ENCOUNTER', 'DATE', 'DESCRIPTION', 'VALUE'], chunksize=CHUNK):
        parts.append(c[c.ENCOUNTER.isin(cohort) | (c.PATIENT.isin(subjects) & c.DESCRIPTION.isin(VITALS))] if WINDOW_HOURS
                     else c[c.ENCOUNTER.isin(cohort)])
    obs = pd.concat(parts, ignore_index=True).rename(columns={'ENCOUNTER': 'hadm_id', 'PATIENT': 'subject_id'})
    obs['DATE'] = pd.to_datetime(obs['DATE'], errors='coerce', utc=True).dt.tz_localize(None)
    v = obs[obs.DESCRIPTION.isin(VITALS)].copy()
    v['ch'] = v.DESCRIPTION.map(VITALS); v['VALUE'] = pd.to_numeric(v.VALUE, errors='coerce')
    if WINDOW_HOURS:   # latest reading per channel up to the window end, from any earlier visit too
        vv = df[['hadm_id', 'subject_id', 'wend']].merge(v[['subject_id', 'DATE', 'ch', 'VALUE']], on='subject_id')
        vv = vv[vv.DATE <= vv.wend].sort_values('DATE')
        vit = vv.groupby(['hadm_id', 'ch'])['VALUE'].last().unstack().reindex(columns=CHANNELS)
    else:
        vit = v.groupby(['hadm_id', 'ch'])['VALUE'].mean().unstack().reindex(columns=CHANNELS)
    vit.columns = [f"vital_{c}" for c in vit.columns]
    o = obs[obs.hadm_id.isin(cohort)].merge(df[['hadm_id', 'admittime', 'wend']], on='hadm_id')
    o = o[(o.DATE >= o.admittime) & (o.DATE <= o.wend) & ~o.DESCRIPTION.isin(LAB_SKIP)]
    lab_ids = set(o.hadm_id)
    save({'vit': vit, 'lab_ids': lab_ids}, "vitals_labs.pkl"); del obs
    print(f"observations done in {time.time()-t0:.0f}s")
d = load("vitals_labs.pkl"); vit, lab_ids = d['vit'], d['lab_ids']
print(f"Stays with any vital: {len(vit):,} / {len(df):,} | with lab activity: {len(lab_ids):,}")
print(vit.notna().mean().round(3).to_string())

In [ ]:
# Cell 4 - medication tokens (NAME_DOSE, time-ordered) + medication activity flag
FORM = ["TABLET", "CAPSULE", "INJECTION", "ORAL SOLUTION", "SOLUTION", "SUSPENSION", "SYRINGE", "VIAL", "PATCH",
        "BAG", "SPRAY", "CREAM", "OINTMENT", "SUPPOSITORY", "DROPPER", "DISKUS", "DEVICE", "LOZENGE", "TUBE",
        "ELIXIR", "LIQUID", "NEBULIZER", "ORAL", "POWDER", "GRANULES", "CONCENTRATE", "PREFILLED"]
def clean_drug_name(d):
    d = d.upper().strip(); d = re.sub(r'\*(NF|NON-FORMULARY|OB|DC)\*\s*', '', d)
    d = re.sub(r'\s*\(.*?\)', '', d); d = re.sub(r'\s*\d+(\.\d+)?\s*%$', '', d)
    for f in FORM: d = d.replace(f, '')
    return d.strip() or "UNKNOWN"
def dose(d):
    m = re.search(r'(\d+\.?\d*)\s*(mg|mcg|g|ml|iu|units|meq|%)', d, re.I)
    if m: return f"{m.group(1)}{m.group(2)}".upper()
    m = re.search(r'(\d+\.?\d*)\s*/\s*(\d+\.?\d*)\s*(mg|mcg|ml|units)', d, re.I)
    return f"{m.group(1)}/{m.group(2)}{m.group(3)}".upper() if m else "UNKNOWN"
def med_token(d):
    if not isinstance(d, str): return "UNKNOWN"
    n = re.sub(r'\d+\.?\d*\s*(MG|MCG|G|ML|IU|MEQ|UNITS)', '', clean_drug_name(d)).strip()
    n = re.sub(r'\s+', '_', n).strip('_')
    return f"{n}_{dose(d)}" if n and n != "UNKNOWN" else "UNKNOWN"

if not done("meds.pkl"):
    cols = ['START', 'STOP', 'ENCOUNTER', 'PATIENT', 'DESCRIPTION']
    parts = [(c[c.PATIENT.isin(subjects)] if WINDOW_HOURS else c[c.ENCOUNTER.isin(cohort)])
             for c in pd.read_csv(f"{RAW}/medications.csv", usecols=cols, chunksize=CHUNK)]
    raw = pd.concat(parts, ignore_index=True).rename(columns={'PATIENT': 'subject_id'})
    for c in ('START', 'STOP'): raw[c] = pd.to_datetime(raw[c], errors='coerce', utc=True).dt.tz_localize(None)
    raw['tok'] = raw.DESCRIPTION.apply(med_token); raw = raw[raw.tok != 'UNKNOWN']
    if WINDOW_HOURS:   # active at admission (started before, not stopped) + started within the window
        m2 = raw.merge(df[['hadm_id', 'subject_id', 'admittime', 'wend']], on='subject_id')
        active = (m2.START < m2.admittime) & (m2.STOP.isna() | (m2.STOP >= m2.admittime))
        started = (m2.START >= m2.admittime) & (m2.START <= m2.wend)
        med = m2[active | started].copy()
        med_ids = set(m2.loc[started, 'hadm_id'])
    else:
        med = raw.rename(columns={'ENCOUNTER': 'hadm_id'})
        m2 = med.merge(df[['hadm_id', 'admittime', 'dischtime']], on='hadm_id')
        med_ids = set(m2.loc[(m2.START >= m2.admittime) & (m2.START <= m2.dischtime), 'hadm_id'])
    med = med.sort_values(['hadm_id', 'START'])
    save({'enc_meds': med.groupby('hadm_id')['tok'].apply(list).to_dict(), 'med_ids': med_ids,
          'pat_seq': med.sort_values(['subject_id', 'START']).groupby('subject_id')['tok'].apply(list).to_dict()}, "meds.pkl")
M = load("meds.pkl"); enc_meds, med_ids = M['enc_meds'], M['med_ids']
print(f"Stays with medications: {len(enc_meds):,} | med started in stay: {len(med_ids):,}")

In [ ]:
# Cell 5 - conditions (at the stay + already active at admission) and procedures
if not done("cond_proc.pkl"):
    cond = pd.read_csv(f"{RAW}/conditions.csv", usecols=['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION'])
    for c in ('START', 'STOP'): cond[c] = pd.to_datetime(cond[c], errors='coerce', utc=True).dt.tz_localize(None)
    cond = cond.rename(columns={'PATIENT': 'subject_id', 'ENCOUNTER': 'cond_enc'})
    at_stay = cond[cond.cond_enc.isin(cohort)][['cond_enc', 'CODE', 'START']].rename(columns={'cond_enc': 'hadm_id'})
    at_stay = at_stay.merge(df[['hadm_id', 'wend']], on='hadm_id')
    at_stay = at_stay[at_stay.START <= at_stay.wend][['hadm_id', 'CODE', 'START']]
    c2 = cond[cond.subject_id.isin(set(df.subject_id))].merge(df[['hadm_id', 'subject_id', 'admittime']], on='subject_id')
    active = c2[(c2.START < c2.admittime) & (c2.STOP.isna() | (c2.STOP >= c2.admittime))][['hadm_id', 'CODE', 'START']]
    cevents = pd.concat([at_stay, active]).drop_duplicates(['hadm_id', 'CODE']).sort_values(['hadm_id', 'START'])
    cevents['CODE'] = cevents.CODE.astype(str)
    proc = pd.concat([c[c.ENCOUNTER.isin(cohort)] for c in
                      pd.read_csv(f"{RAW}/procedures.csv", usecols=['START', 'ENCOUNTER', 'CODE', 'DESCRIPTION'], chunksize=CHUNK)])
    proc = proc.rename(columns={'ENCOUNTER': 'hadm_id'})
    proc['START'] = pd.to_datetime(proc.START, errors='coerce', utc=True).dt.tz_localize(None)
    proc = proc.merge(df[['hadm_id', 'admittime', 'wend']], on='hadm_id')
    proc = proc[(proc.START >= proc.admittime - pd.Timedelta(hours=1)) & (proc.START <= proc.wend)].drop(columns=['admittime', 'wend'])
    save({'cevents': cevents, 'proc': proc.sort_values(['hadm_id', 'START'])}, "cond_proc.pkl"); del cond, c2
CP = load("cond_proc.pkl"); cevents, proc = CP['cevents'], CP['proc']
print(f"Condition events {len(cevents):,} ({cevents.CODE.nunique()} codes) | procedure rows {len(proc):,} "
      f"({proc.DESCRIPTION.nunique()} unique descriptions)")

In [ ]:
# Cell 6 - OPTIONAL: Groq LLM mappings (Synthea only; never use on MIMIC). Key from Colab Secrets.
import requests
def groq(system, user, model, max_tokens):
    from google.colab import userdata
    key = userdata.get('GROQ_API_KEY')
    r = requests.post("https://api.groq.com/openai/v1/chat/completions", timeout=30,
                      headers={"Authorization": f"Bearer {key}"},
                      json={"model": model, "temperature": 0.0, "max_tokens": max_tokens,
                            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]})
    r.raise_for_status(); return r.json()['choices'][0]['message']['content'].strip().replace('```json', '').replace('```', '')

proc_map = load("desc_to_icd10pcs.pkl") if done("desc_to_icd10pcs.pkl") else {}
if USE_LLM_PROCEDURES:
    SYS = ('Return ONLY JSON {"code": "<ICD-10-PCS>"} with the single most appropriate ICD-10-PCS code '
           'for the procedure name. If none fits, use the closest ICD-10-CM or Z code.')
    for i, d in enumerate([d for d in proc.DESCRIPTION.unique() if d not in proc_map]):
        try: proc_map[d] = json.loads(groq(SYS, re.sub(r'\s*\((procedure|regime/therapy)\)', '', d), "llama-3.3-70b-versatile", 50))['code']
        except Exception: proc_map[d] = None
        if i % 20 == 19: save(proc_map, "desc_to_icd10pcs.pkl")
        time.sleep(0.5)
    save(proc_map, "desc_to_icd10pcs.pkl")
proc['pcode'] = proc.DESCRIPTION.map(proc_map) if proc_map else None
proc['pcode'] = proc['pcode'].fillna('SCT_' + proc.CODE.astype(str))   # fallback: SNOMED procedure code
print(f"Procedures with ICD-10-PCS: {proc.pcode.str.startswith('SCT_').eq(False).mean():.1%} (rest use SNOMED)")

In [ ]:
# Cell 7 - notes: unpack, assign each dated segment to a stay (+/- 3 days, one-to-one), keep key sections
if not os.path.exists(NOTES_DIR):
    os.makedirs("/content/notes_local", exist_ok=True)
    !tar -xzf "{NOTES_TAR}" -C /content/notes_local
files = {f.replace('.txt', '').split('_')[-1]: os.path.join(NOTES_DIR, f) for f in os.listdir(NOTES_DIR)}
DATE_RE = re.compile(r'^\d{4}-\d{2}-\d{2}', re.M)
HEADER_RE = re.compile(r'^#{1,3}\s*(.+?)\s*$', re.M)
KEEP = {'hpi': r'\b(history of present illness|hpi)\b', 'allergies': r'\ballerg(y|ies)\b',
        'medications': r'\b(medications?|meds)\b', 'assessment_plan': r'\b(assessment|plan)\b',
        'radiology': r'\b(radiology|imaging)\b', 'nursing': r'\bnursing\b'}
KEEP = {k: re.compile(v, re.I) for k, v in KEEP.items()}
SOCIAL = re.compile(r'\bsocial history\b', re.I)

def segments(text):
    m = list(DATE_RE.finditer(text)); out = []
    for i, x in enumerate(m):
        try: out.append((pd.Timestamp(x.group()), text[x.start(): m[i+1].start() if i+1 < len(m) else len(text)]))
        except Exception: pass
    return out
def keep_sections(text):
    hs = list(HEADER_RE.finditer(text)); out = {}
    for i, h in enumerate(hs):
        name = h.group(1)
        if SOCIAL.search(name): continue
        body = text[h.end(): hs[i+1].start() if i+1 < len(hs) else len(text)].strip()
        for k, p in KEEP.items():
            if p.search(name): out[k] = (out.get(k, '') + ' ' + body).strip(); break
    return out

if not done("encounter_notes.pkl"):
    notes, nofile = {}, 0
    for sid, g in df.groupby('subject_id'):
        if sid not in files: nofile += len(g); continue
        segs = segments(open(files[sid], encoding='utf-8', errors='ignore').read())
        if WINDOW_HOURS:   # only the note dated on the admission day (up to the window)
            last_day = max(WINDOW_HOURS // 24 - 1, 0)
            cand = sorted((abs(sd - a.normalize()), e, s) for e, a in enumerate(g.admittime) for s, (sd, _) in enumerate(segs)
                          if 0 <= (sd - a.normalize()).days <= last_day)
        else:
            cand = sorted((abs(sd - a), e, s) for e, a in enumerate(g.admittime) for s, (sd, _) in enumerate(segs)
                          if abs(sd - a) <= timedelta(days=3))
        ue, us = set(), set()
        for _, e, s in cand:
            if e in ue or s in us: continue
            notes[g.hadm_id.iloc[e]] = keep_sections(segs[s][1]); ue.add(e); us.add(s)
    save(notes, "encounter_notes.pkl"); print(f"stays in patients with no note file: {nofile}")
notes = load("encounter_notes.pkl")
print(f"Stays with a matched note: {len(notes):,} / {len(df):,}")
print(pd.Series([k for v in notes.values() for k in v]).value_counts().to_string())

In [ ]:
# Cell 8 - note cleaning: pronoun removal (HPI), optional scispaCy abbreviations, tags, stop words -> tokens
import nltk; nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOP = set(stopwords.words('english')) | {'were', 'was', 'conducted', 'hearing', 'examination', 'examinations',
        'employment', 'patient', 'presents', 'history', 'reports', 'denies', 'mg', 'tablet'}
PRON = re.compile(r'\b(he|she|him|her|his|hers|himself|herself|they|them|their|theirs|themselves)\b', re.I)
PHI = [(re.compile(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b'), ' DATE '), (re.compile(r'\b\d{3}-\d{3}-\d{4}\b'), ' PHONE '),
       (re.compile(r'\b\d{3}-\d{2}-\d{4}\b'), ' SSN '), (re.compile(r'\b(mr|mrs|ms|dr)\.?\s+[A-Z][a-z]+\b', re.I), ' NAME ')]
AGE = re.compile(r'\b(\d{1,3})\s*(?:years?[\s-]*old|y\.?o\.?|yo)\b', re.I)
PROTECT = {'nkda', 'pcn', 'asa', 'nsaid', 'sulfa', 'date', 'phone', 'ssn', 'name', 'lab_high', 'lab_low'}

if USE_SCISPACY:
    !pip -q install spacy==3.7.4 && pip -q install scispacy==0.5.4 --no-deps && pip -q install conllu nmslib-metabrainz
    !pip -q install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
    import spacy; from scispacy.abbreviation import AbbreviationDetector
    nlp = spacy.load("en_core_sci_sm"); nlp.add_pipe("abbreviation_detector")
def expand(t):
    if not USE_SCISPACY or not t: return t
    doc = nlp(t)
    for a in sorted(doc._.abbreviations, key=lambda a: len(a.text), reverse=True): t = t.replace(a.text, str(a._.long_form))
    return t
def tokenize(t):
    for p, r in PHI: t = p.sub(r, t)
    t = AGE.sub(lambda m: f" AGE_{int(m.group(1))//10*10}_{int(m.group(1))//10*10+9} " if int(m.group(1)) <= 120 else m.group(0), t)
    t = re.sub(r'\b\d+(\.\d+)?\s*\(?H\)?\b', ' LAB_HIGH ', t); t = re.sub(r'\b\d+(\.\d+)?\s*\(?L\)?\b', ' LAB_LOW ', t)
    out = []
    for w in t.split():
        w = w.strip('.,;:()[]{}*#-'); lw = w.lower()
        if not w: continue
        if w.startswith('AGE_'): out.append(w); continue
        if lw in PROTECT: out.append(lw); continue
        if lw in STOP or len(lw) < 3 or not re.search('[a-z]', lw): continue
        out.append(lw)
    return out
if not done("encounter_notes_tokens.pkl"):
    t0 = time.time(); toks = {}
    for i, (hid, secs) in enumerate(notes.items()):
        if 'hpi' in secs: secs = {**secs, 'hpi': re.sub(r'\s+', ' ', PRON.sub('', secs['hpi']))}
        toks[hid] = [w for k in sorted(secs) for w in tokenize(expand(secs[k]))]
        if i % 5000 == 4999: print(f"  {i+1:,} notes | {time.time()-t0:.0f}s")
    save(toks, "encounter_notes_tokens.pkl")
note_toks = load("encounter_notes_tokens.pkl")
print(f"Token lists: {len(note_toks):,} | median length {np.median([len(v) for v in note_toks.values()]):.0f}")

In [ ]:
# Cell 9 - OPTIONAL: Groq note -> ICD-10-CM codes (adds to the condition sequence)
llm_codes = load("llm_icd_codes.pkl") if done("llm_icd_codes.pkl") else {}
if USE_LLM_NOTES:
    SYS = ("You are a clinical coding assistant. Read the diagnoses and findings described in plain English and "
           "assign the single most appropriate ICD-10-CM code to each. Ignore tags like (disorder) or (finding). "
           'Return ONLY a JSON array of ICD-10-CM codes, e.g. ["J18.9", "I10"]. If none, return [].')
    todo = [h for h in notes if h not in llm_codes]; print(f"LLM notes to process: {len(todo):,}")
    for i, h in enumerate(todo):
        txt = ' '.join(f"{k}: {v}" for k, v in notes[h].items())[:3500]
        try:
            codes = json.loads(groq(SYS, txt, "llama-3.1-8b-instant", 200))
            llm_codes[h] = list(dict.fromkeys(c[:3] + '.' + c[3:] if re.match(r'^[A-Z]\d{3,}$', c) else c
                                              for c in map(lambda x: str(x).strip(), codes)))
        except Exception: llm_codes[h] = []
        if i % 50 == 49: save(llm_codes, "llm_icd_codes.pkl")
        time.sleep(0.5)
    save(llm_codes, "llm_icd_codes.pkl")
print(f"Stays with LLM codes: {sum(1 for v in llm_codes.values() if v):,}")

In [ ]:
# Cell 10 - per-stay sequences, Word2Vec per modality fit on TRAIN patients only
from gensim.models import Word2Vec
order = df.sort_values('admittime')
enc_cond = cevents.groupby('hadm_id')['CODE'].apply(list).to_dict()
for h, cs in llm_codes.items():
    if cs: enc_cond[h] = enc_cond.get(h, []) + ['ICD_' + c for c in cs]
enc_proc = proc.groupby('hadm_id')['pcode'].apply(lambda s: [str(x) for x in s]).to_dict()
MODS = {'conditions': (enc_cond, 1, 10), 'procedures': (enc_proc, 1, 10), 'notes': (note_toks, 2, 5), 'medications': (enc_meds, 1, 10)}
w2v = {}
tr = order[order.subject_id.isin(train_subj)]
for name, (enc_map, min_count, epochs) in MODS.items():
    sents = [s for s in (sum((enc_map.get(h, []) for h in g.hadm_id), []) for _, g in tr.groupby('subject_id')) if s]
    path = f"{OUT}/word2vec_{name}.model"
    w2v[name] = Word2Vec.load(path) if os.path.exists(path) else Word2Vec(sents, vector_size=W2V_DIM, window=5,
                    min_count=min_count, sg=1, epochs=epochs, seed=SEED, workers=4)
    w2v[name].save(path)
    print(f"{name:12s}: {len(sents):,} train patients | vocab {len(w2v[name].wv):,}")

In [ ]:
# Cell 11 - assemble the 524-number vector per stay; save a flat table (for the generator) and patient sequences (for RAIM)
def pool(tokens, m):
    v = [m.wv[t] for t in tokens if t in m.wv]
    return np.mean(v, 0).astype('float32') if v else np.zeros(W2V_DIM, 'float32')
X = df[['hadm_id', 'subject_id', 'split', 'admittime', 'target', 'is_plos', 'anchor_age', 'gender', 'race',
        'ethnicity', 'encounter_type', 'encounter_reason']].merge(vit, left_on='hadm_id', right_index=True, how='left')
VCOLS = [f"vital_{c}" for c in CHANNELS]
X['has_vitals'] = X[VCOLS].notna().any(axis=1).astype('int8'); X[VCOLS] = X[VCOLS].fillna(0)
X['gender_enc'] = (X.gender == 'M').astype('int8')
X['race_enc'] = pd.Categorical(X.race).codes; X['ethnicity_enc'] = pd.Categorical(X.ethnicity).codes
X['lab_active'] = X.hadm_id.isin(lab_ids).astype('int8'); X['med_active'] = X.hadm_id.isin(med_ids).astype('int8')
blocks = {}
for name, (enc_map, _, _) in MODS.items():
    E = np.stack([pool(enc_map.get(h, []), w2v[name]) for h in X.hadm_id])
    blocks[name] = E
    X = pd.concat([X, pd.DataFrame(E, columns=[f"{name[:4]}_{i}" for i in range(W2V_DIM)], index=X.index)], axis=1)
PROFILE = ['anchor_age', 'gender_enc', 'race_enc', 'ethnicity_enc', 'has_vitals'] + VCOLS
EMB = [c for n in MODS for c in X.columns if c.startswith(n[:4] + '_')]
FEAT524 = PROFILE + EMB + ['lab_active', 'med_active']
assert len(FEAT524) == 524, len(FEAT524)
X.to_parquet(f"{OUT}/encounters_524.parquet", index=False)

seqs = {}
for sid, g in X.sort_values('admittime').groupby('subject_id'):
    seqs[sid] = {'X': g[FEAT524].values.astype('float32'), 'Y': g.target.values.astype('float32'),
                 'hadm_ids': g.hadm_id.tolist(), 'split': g.split.iloc[0], 'n_encounters': len(g)}
save(seqs, "sequences_patient_level.pkl")
json.dump({'feat524': FEAT524, 'profile': PROFILE, 'blocks': {n: [c for c in X.columns if c.startswith(n[:4] + '_')] for n in MODS},
           'threshold': THR, 'window_hours': WINDOW_HOURS, 'llm_procedures': USE_LLM_PROCEDURES, 'llm_notes': USE_LLM_NOTES, 'scispacy': USE_SCISPACY},
          open(f"{OUT}/features_meta.json", "w"), indent=1)
print(f"Saved encounters_524.parquet ({len(X):,} stays x 524 features) and sequences for {len(seqs):,} patients")

In [ ]:
# Cell 12 - coverage report (what each modality contributes, PLOS vs non-PLOS)
cov = pd.DataFrame({
    'has_vitals': X.has_vitals, 'lab_active': X.lab_active, 'med_active': X.med_active,
    **{f"has_{n}": (np.abs(blocks[n]).sum(1) > 0).astype(int) for n in MODS}, 'is_plos': X.is_plos})
rep = cov.groupby('is_plos').mean().T.rename(columns={0: 'non-PLOS', 1: 'PLOS'})
rep['all'] = cov.drop(columns='is_plos').mean()
print("Share of stays where each modality is present:"); print(rep.round(3).to_string())
rep.to_csv(f"{OUT}/modality_coverage.csv")